In [1]:
import PyKDL
from geometry_msgs.msg import TransformStamped

def transform_to_kdl_rotation(transform: TransformStamped):
    """Convert a geometry_msgs TransformStamped message to a PyKDL Rotation matrix."""
    return PyKDL.Rotation.Quaternion(
        transform.transform.rotation.x,
        transform.transform.rotation.y,
        transform.transform.rotation.z,
        transform.transform.rotation.w
    )

def compute_angular_velocity_body(omega_A, transform):
    """
    Transforms the angular velocity from the inertial frame (A) to the body frame (B).

    :param omega_A: Angular velocity in the inertial frame (PyKDL.Vector).
    :param transform: Transform from A to B (geometry_msgs.msg.TransformStamped).
    :return: Angular velocity in the body frame (PyKDL.Vector).
    """
    R = transform_to_kdl_rotation(transform)  # Get the rotation matrix R_A^B
    omega_B = R * omega_A  # Transform angular velocity to body frame
    return omega_B

# === TEST CASE ===
# Fake transform from "inertial_frame" to "body_frame"
fake_transform = TransformStamped()
fake_transform.transform.rotation.x = 0.0  # Example quaternion (no rotation)
fake_transform.transform.rotation.y = 0.0
fake_transform.transform.rotation.z = 0.0
fake_transform.transform.rotation.w = 1.0  # Identity rotation (R = I)

# Angular velocity in inertial frame A
omega_A = PyKDL.Vector(0, 0, 1.0)  # 1 rad/s about z-axis in A

# Compute angular velocity in body frame B
omega_B = compute_angular_velocity_body(omega_A, fake_transform)

print("Angular Velocity in Inertial Frame (A):", omega_A)
print("Angular Velocity in Body Frame (B):", omega_B)  # Should be zero if R evolves correctly


Angular Velocity in Inertial Frame (A): [           0,           0,           1]
Angular Velocity in Body Frame (B): [           0,           0,           1]


In [2]:
import pandas as pd
from pandas import DataFrame
from typing import Annotated
import numpy as np

PoseDF = Annotated[pd.DataFrame, {"x": float, "y": float, "z": float, "qx": float, "qy": float, "qz": float, "qw": float}]

df: PoseDF = pd.DataFrame({
    'x': np.random.rand(5),
    'y': np.random.rand(5),
    'z': np.random.rand(5),
    'qx': np.random.rand(5),
    'qy': np.random.rand(5),
    'qz': np.random.rand(5),
    'foo1': np.random.rand(5),
    'foo': np.random.rand(5),
})

x: PoseDF = df


In [3]:
from tf2_kdl import transform_to_kdl
# http://docs.ros.org/en/diamondback/api/kdl/html/python/geometric_primitives.html?highlight=twist#PyKDL.Twist
def do_transform_twist(twist, transform):
    """
    Apply a transform in the form of a geometry_msgs message to a PyKDL Twist.

    :param twist: The PyKDL twist to transform.
    :type twist: PyKDL.Twist
    :param transform: The transform to apply.
    :type transform: geometry_msgs.msg.TransformStamped
    :return: The transformed PyKDL twist.
    :rtype: PyKDL.Twist
    """
    res: PyKDL.Frame = transform_to_kdl(transform) * twist
    # res.header = transform.header
    return res

In [4]:
import pandas as pd

# from tf2_kdl import do_transform_twist
from geometry_msgs.msg import TransformStamped

def transform_twist_dataframe(df, transform):
    """
    Transforms a DataFrame of twists (linear xyz + angular rpy) into another frame using PyKDL.

    :param df: Pandas DataFrame with columns ['vx', 'vy', 'vz', 'wx', 'wy', 'wz']
               representing a twist (linear and angular velocity).
    :param transform: Transform from the original frame to the target frame (geometry_msgs.msg.TransformStamped).
    :return: New DataFrame with transformed twists.
    """
    transformed_data = []

    # For each Twist msg, make a new KDL Twist version of it.
    for _, row in df.iterrows():
        # Create PyKDL Twist (linear velocity, angular velocity)
        twist = PyKDL.Twist(
            PyKDL.Vector(row['vx'], row['vy'], row['vz']),  # Linear velocity
            PyKDL.Vector(row['wx'], row['wy'], row['wz'])   # Angular velocity
        )

        transformed_twist = do_transform_twist(twist, transform)

        # Store transformed values
        transformed_data.append([
            transformed_twist.vel[0], transformed_twist.vel[1], transformed_twist.vel[2],
            transformed_twist.rot[0], transformed_twist.rot[1], transformed_twist.rot[2]
        ])

    # Create a new DataFrame with the transformed data
    new_df = pd.DataFrame(transformed_data, columns=['vx', 'vy', 'vz', 'wx', 'wy', 'wz'], index=df.index)
    return new_df


In [6]:
from std_msgs.msg import Header
import rclpy

# Create a sample DataFrame with twists
timestamps = np.linspace(0, 10, 5)  # 5 timestamps from 0s to 10s
df = pd.DataFrame({
    'vx': np.random.rand(5),
    'vy': np.random.rand(5),
    'vz': np.random.rand(5),
    'wx': np.random.rand(5),
    'wy': np.random.rand(5),
    'wz': np.random.rand(5),
}, index=pd.to_datetime(timestamps, unit='s'))

# Define a fake transformation from one frame to another
fake_transform = TransformStamped()
header = Header()
header.stamp = 100  # Use current time
header.frame_id = "vicon"
fake_transform.header = header
fake_transform.transform.translation.x = 1.0
fake_transform.transform.translation.y = 2.0
fake_transform.transform.translation.z = 3.0
fake_transform.transform.rotation.x = 0.0
fake_transform.transform.rotation.y = 0.0
fake_transform.transform.rotation.z = 0.7071068  # 90-degree rotation about Z
fake_transform.transform.rotation.w = 0.7071068  # (cos(45°))

# Transform the DataFrame
new_df = transform_twist_dataframe(df, fake_transform)
